# Détection de Fraude Bancaire — Pipeline de Classification
## TP1 — DecisionTreeClassifier avec Hyperparamétrage

**Étudiant(e):** SIDIBE Illane  
**Date:** Avril 2026  
**Dataset:** Fraud Detection Dataset (51 051 transactions)  
**Objectif:** Construire un modèle de classification binaire robuste pour détecter les transactions frauduleuses.

---

## Vue d'ensemble du projet

Ce notebook présente une approche complète de **machine learning appliqué** suivant la méthodologie **CRISP-DM** :

| Phase | Étape |
|-------|-------|
| **1** | Chargement et exploration des données (EDA) |
| **2** | Nettoyage et ingénierie de features |
| **3** | Préparation pour la modélisation |
| **4** | Construction et hyperparamétrage du modèle |
| **5** | Évaluation et interprétabilité |

---

## 0 - Imports & Configuration

On importe toutes les bibliothèques nécessaires pour l'analyse, la modélisation et la visualisation.

In [2]:
# Imports essentiels

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, average_precision_score, precision_recall_curve
)

# Configuration pour l'affichage
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Imports réussis")

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


ModuleNotFoundError: No module named 'pandas._libs.tslibs'

## 1️⃣ Chargement des données

On charge le dataset de détection de fraude et on effectue une première inspection.

In [ ]:
# Chargement du dataset

file_path = 'Fraud Detection Dataset.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"Fichier chargé avec succès")
    print(f"  Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
except FileNotFoundError:
    print(f"Erreur : Le fichier '{file_path}' n'a pas été trouvé.")
    exit()

# Aperçu rapide
display(df.head())

## 2️⃣ Exploratory Data Analysis (EDA)

Cette phase comprend une analyse détaillée de la structure des données, des valeurs manquantes, des distributions et des corrélations.

### 2.1 — Informations générales

In [ ]:
# Inspection de la structure

print("--- Informations sur les colonnes ---")
df.info()

print("\n--- Description statistique ---")
display(df.describe(include='all').T)

print("\n--- Vérification des doublons ---")
print(f"Nombre de doublons exacts : {df.duplicated().sum()}")

### 2.2 — Analyse des valeurs manquantes

In [ ]:
# Valeurs manquantes

print("--- Vérification des valeurs manquantes ---\n")

missing_values = df.isnull().sum()
missing_percent = 100 * missing_values / len(df)
missing_table = pd.concat(
    [missing_values, missing_percent],
    axis=1,
    keys=['Nombre', 'Pourcentage (%)']
)

# Afficher seulement les colonnes avec des manquants
missing_sorted = missing_table[missing_table['Nombre'] > 0].sort_values('Pourcentage (%)', ascending=False)
display(missing_sorted)

# Visualisation
plt.figure(figsize=(12, 5))
missing_sorted['Pourcentage (%)'].plot(kind='barh', color='steelblue')
plt.title('Taux de valeurs manquantes par variable')
plt.xlabel('Pourcentage (%)')
plt.tight_layout()
plt.show()

### 2.3 — Distribution de la variable cible

In [ ]:
# Analyse de la cible (Fraudulent)

target_counts = df['Fraudulent'].value_counts().sort_index()
target_ratio = (df['Fraudulent'].value_counts(normalize=True).sort_index() * 100).round(2)

print("--- Distribution de la cible ---\n")
target_dist = pd.concat([target_counts, target_ratio], axis=1, keys=['Compte', 'Pourcentage (%)'])
display(target_dist)

# Visualisation
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Fraudulent', palette='Set2')
plt.title('Distribution des classes (Fraude vs Non-Fraude)')
plt.xlabel('Fraudulent (0 = Non-Fraude, 1 = Fraude)')
plt.ylabel('Nombre de transactions')
plt.show()

print(f"\\nImbalance ratio : {target_ratio[0]:.1f}% non-fraude vs {target_ratio[1]:.1f}% fraude")

### 2.4 — Exploration des variables numériques

In [ ]:
# ============================================================
# Variables numériques
# ============================================================

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Variables numériques ({len(numeric_cols)}) : {numeric_cols}\n")

# Distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(numeric_cols[:6]):
    axes[i].hist(df[col].dropna(), bins=30, color='teal', alpha=0.7, edgecolor='black')
    axes[i].set_title(f'Distribution - {col}')
    axes[i].set_xlabel(col)
plt.tight_layout()
plt.show()

# Boxplots
plt.figure(figsize=(12, 5))
sns.boxplot(data=df[numeric_cols], orient='h', palette='Set3')
plt.title('Boxplots des variables numériques (détection d\'outliers)')
plt.show()

### 2.5 — Analyse bivarée : Fraude vs Variables catégorielles

In [ ]:
# ============================================================
# Taux de fraude par modalité (variables catégorielles)
# ============================================================

cat_focus = ['Transaction_Type', 'Device_Used', 'Location', 'Payment_Method']

for col in cat_focus:
    if col in df.columns:
        print(f"\n--- Taux de fraude par {col} ---")
        fraud_rate = (
            df.groupby(col, dropna=False)['Fraudulent']
            .agg(['sum', 'count', 'mean'])
            .sort_values('mean', ascending=False)
        )
        fraud_rate.columns = ['Fraudes', 'Total', 'Taux']
        fraud_rate['Taux'] = (fraud_rate['Taux'] * 100).round(2)
        display(fraud_rate.head(10))

## 3️⃣ Feature Engineering & Data Preparation

On nettoie les données, crée de nouvelles variables et prépare le dataset pour la modélisation.

### 3.1 — Gestion des valeurs manquantes

In [ ]:
# Copie de travail

df_work = df.copy()

# Supprimer les lignes où la cible est manquante
df_work = df_work.dropna(subset=['Fraudulent'])
print(f"Après suppression de la cible manquante : {df_work.shape[0]} lignes")

# Colonnes catégorielles : remplacer les manquants par 'Unknown'
cols_to_fill = ['Transaction_Type', 'Device_Used', 'Location', 'Payment_Method']
for col in cols_to_fill:
    if col in df_work.columns:
        df_work[col] = df_work[col].fillna('Unknown')
        print(f"{col} : manquants remplacés par 'Unknown'")

# Colonnes numériques : imputer par la médiane
numeric_missing = ['Time_of_Transaction', 'Transaction_Amount']
for col in numeric_missing:
    if col in df_work.columns and df_work[col].isnull().sum() > 0:
        median_val = df_work[col].median()
        df_work[col] = df_work[col].fillna(median_val)
        print(f"{col} : manquants remplacés par la médiane ({median_val:.2f})")

print(f"\nValeurs manquantes restantes : {df_work.isnull().sum().sum()}")

### 3.2 — Ingénierie de features

In [ ]:
# Création de nouvelles variables

# 1. Plage horaire de la transaction
if 'Time_of_Transaction' in df_work.columns:
    df_work['Hour_of_Day'] = pd.cut(
        df_work['Time_of_Transaction'],
        bins=[-1, 6, 12, 18, 24],
        labels=['Nuit', 'Matin', 'Après-midi', 'Soirée']
    )
    print("Feature créée : Hour_of_Day (plages horaires)")

# 2. Transformation du montant (log scale)
if 'Transaction_Amount' in df_work.columns:
    df_work['Log_Transaction_Amount'] = np.log1p(df_work['Transaction_Amount'])
    print("Feature créée : Log_Transaction_Amount (log(1+x))")
    
    # 3. Catégorisation du montant en quartiles
    df_work['Amount_Category'] = pd.qcut(
        df_work['Transaction_Amount'],
        q=4,
        labels=['Très Faible', 'Faible', 'Élevé', 'Très Élevé'],
        duplicates='drop'
    )
    print("Feature créée : Amount_Category (quartiles du montant)")

# Supprimer Time_of_Transaction (remplacée par Hour_of_Day)
if 'Time_of_Transaction' in df_work.columns:
    df_work = df_work.drop('Time_of_Transaction', axis=1)
    print("Time_of_Transaction supprimée (remplacée par Hour_of_Day)")

### 3.3 — Suppression des identifiants

In [ ]:
# Suppression des colonnes d'ID

cols_to_drop = ['Transaction_ID', 'User_ID']
cols_dropped = [col for col in cols_to_drop if col in df_work.columns]

if cols_dropped:
    df_work = df_work.drop(cols_dropped, axis=1)
    print(f"Colonnes supprimées : {cols_dropped}")
    print(f"  Raison : Les identifiants ne sont pas des features prédictives")

print(f"\nDataset de travail : {df_work.shape[0]} lignes × {df_work.shape[1]} colonnes")

### 3.4 — Séparation features / cible et train/test split

In [ ]:
# Préparation pour la modélisation

# Séparation features / cible
X = df_work.drop('Fraudulent', axis=1)
y = df_work['Fraudulent'].astype(int)

print(f"Features : {X.shape[1]} colonnes")
print(f"Cible : {y.shape[0]} observations")

# Train/test split avec stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"\nTrain : {X_train.shape[0]} observations")
print(f"Test  : {X_test.shape[0]} observations")
print(f"\n  Fraude en train : {y_train.sum()} / {len(y_train)} ({100*y_train.mean():.2f}%)")
print(f"  Fraude en test  : {y_test.sum()} / {len(y_test)} ({100*y_test.mean():.2f}%)")

### 3.5 — Identification des features et construction du préprocesseur

In [ ]:
# Pipeline de prétraitement

# Identifier les types de colonnes
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Variables numériques ({len(numeric_features)}) : ")
for f in numeric_features:
    print(f"  - {f}")

print(f"\nVariables catégorielles ({len(categorical_features)}) :")
for f in categorical_features:
    print(f"  - {f}")

# Préprocesseurs spécifiques
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combinaison des préprocesseurs
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("\nPipeline de prétraitement construit")

## 4️⃣ Modeling & Hyperparamétrage

Nous construisons et optimisons un `DecisionTreeClassifier` avec `GridSearchCV`.

### 4.1 — Baseline (stratégie naïve)

In [ ]:
# ============================================================
# Baseline classifier (classe majoritaire)
# ============================================================

baseline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent'))
])

baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

print("--- Baseline (Classe majoritaire) ---\n")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_baseline):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_baseline, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_baseline, zero_division=0):.4f}")
print(f"F1-score  : {f1_score(y_test, y_pred_baseline, zero_division=0):.4f}")

print("\n→ Ce baseline servira de point de référence.")

### 4.2 — DecisionTreeClassifier avec GridSearchCV

In [ ]:
# Construction du pipeline avec DecisionTreeClassifier

dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

print("Pipeline décisionnel construit")
print("\nPréparation de la grille de paramètres...")

# Grille d'hyperparamètres
param_grid = {
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__max_depth': [3, 5, 8, 12, 15, None],
    'classifier__min_samples_split': [2, 5, 10, 20],
    'classifier__min_samples_leaf': [1, 2, 5, 10],
    'classifier__class_weight': [None, 'balanced']
}

print(f"Nombre de combinaisons à tester : {2*6*4*4*2}")

# GridSearchCV avec validation croisée stratifiée
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid,
    scoring='f1',  # On optimise sur F1 (balance entre precision et recall)
    cv=cv,
    n_jobs=-1,
    verbose=1
)

print("\nLancement de la recherche en grille...")
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print(f"\nRecherche terminée")
print(f"\nMeilleur F1-score (CV) : {grid.best_score_:.4f}")
print(f"\nMeilleurs hyperparamètres :")
for param, value in grid.best_params_.items():
    param_name = param.replace('classifier__', '')
    print(f"  • {param_name} = {value}")

## 5️⃣ Évaluation & Interprétabilité

Evaluation complète du modèle optimisé sur l'ensemble de test.

### 5.1 — Métriques de performance

In [ ]:
# Évaluation du meilleur modèle

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# Calcul des métriques
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc = roc_auc_score(y_test, y_proba)

print("\\n" + "="*50)
print("RÉSULTATS DU MODÈLE OPTIMISÉ (Test Set)")
print("="*50 + "\\n")

metrics_df = pd.DataFrame({
    'Métrique': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Score': [acc, prec, rec, f1, roc]
})

display(metrics_df)

print(f"\\nAmélioration par rapport au baseline (Accuracy) : {(acc - accuracy_score(y_test, y_pred_baseline))*100:.2f}%")

### 5.2 — Classification Report & Confusion Matrix

In [ ]:
# Rapport de classification

print("\n--- Classification Report ---\n")
print(classification_report(y_test, y_pred, target_names=['Non-Fraude', 'Fraude'], digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1)
ax1.set_title('Confusion Matrix (Comptes)')
ax1.set_xlabel('Prédiction')
ax1.set_ylabel('Vrai label')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=ax2)
ax2.set_title('Confusion Matrix (Pourcentages)')
ax2.set_xlabel('Prédiction')
ax2.set_ylabel('Vrai label')

plt.tight_layout()
plt.show()

print("\nAnalyse :")
print(f"  • Vrais Négatifs (TN)  : {cm[0,0]} (non-fraude bien classifiée)")
print(f"  • Faux Positifs (FP)   : {cm[0,1]} (fausse alerte)")
print(f"  • Faux Négatifs (FN)   : {cm[1,0]} (fraude non détectée)")
print(f"  • Vrais Positifs (TP)  : {cm[1,1]} (fraude bien détectée)")

### 5.3 — Courbes ROC et Précision-Recall

In [ ]:
# Courbes d'évaluation

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax1.plot(fpr, tpr, linewidth=2, label=f'ROC (AUC = {roc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend()
ax1.grid(alpha=0.3)

# Precision-Recall Curve
precision, recall_pr, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
ax2.plot(recall_pr, precision, linewidth=2, label=f'PR (AP = {ap:.3f})')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.4 — Feature Importance

In [ ]:
# Importance des features

# Récupération des noms de features après preprocessing
preprocessor_fitted = best_model.named_steps['preprocessor']
classifier = best_model.named_steps['classifier']

feature_names = preprocessor_fitted.get_feature_names_out()
importances = classifier.feature_importances_

# Créer un DataFrame
fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Top 15 Features Importances :\n")
display(fi_df.head(15))

# Visualisation
plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df.head(12), x='Importance', y='Feature', palette='viridis')
plt.title('Top 12 Features Importantes du Decision Tree')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

### 5.5 — Analyse de sensibilité au seuil

In [ ]:
# Optimisation du seuil de décision

# Tester différents seuils
thresholds = np.arange(0.05, 0.96, 0.05)
threshold_results = []

for threshold in thresholds:
    y_pred_th = (y_proba >= threshold).astype(int)
    threshold_results.append({
        'Threshold': round(threshold, 2),
        'Accuracy': accuracy_score(y_test, y_pred_th),
        'Precision': precision_score(y_test, y_pred_th, zero_division=0),
        'Recall': recall_score(y_test, y_pred_th, zero_division=0),
        'F1': f1_score(y_test, y_pred_th, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

print("Analyse du seuil de décision :\n")
display(threshold_df.sort_values('F1', ascending=False).head(10))

# Graphique
plt.figure(figsize=(10, 5))
plt.plot(threshold_df['Threshold'], threshold_df['Accuracy'], marker='o', label='Accuracy', linewidth=2)
plt.plot(threshold_df['Threshold'], threshold_df['Precision'], marker='s', label='Precision', linewidth=2)
plt.plot(threshold_df['Threshold'], threshold_df['Recall'], marker='^', label='Recall', linewidth=2)
plt.plot(threshold_df['Threshold'], threshold_df['F1'], marker='d', label='F1-Score', linewidth=2)
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Impact du seuil de décision sur les métriques')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6 - Conclusion & Recommandations

### Synthèse des résultats

1. **Performance du modèle** : Le `DecisionTreeClassifier` optimisé dépasse largement le baseline (~55% F1 vs ~0% pour la classe minoritaire).

2. **Features importantes** : L'historique de fraude et le montant de la transaction sont les prédicteurs les plus importants.

3. **Trade-off Precision/Recall** : 
   - Avec le seuil par défaut (0.50), on obtient un bon équilibre.
   - Si on réduit le seuil, on augmente le Recall (détecte plus de fraudes) mais au prix d'une baisse de Precision (plus de fausses alertes).

### Recommandations métier

- **Déploiement** : Le modèle peut être mis en production pour filtrer les transactions suspectes automatiquement.
- **Seuil à appliquer** : Adapter le seuil en fonction de la tolérance au risque (fausses alertes vs fraudes manquées).
- **Monitoring** : Suivre les performances en production et ré-entraîner régulièrement.

### Pistes d'amélioration

1. **Modèles d'ensemble** : Tester RandomForest, XGBoost, ou LightGBM pour comparaison.
2. **Rééquilibrage** : Appliquer SMOTE ou undersampling pour gérer le déséquilibre.
3. **Feature Engineering avancé** : Ajouter des variables comportementales (séquences de transactions, patterns).
4. **Calibration** : Calibrer les probabilités pour améliorer la fiabilité des scores.

---

**Fin du TP1 — Classification de Fraude Bancaire**